In [ ]:
import numpy as np
import pandas as pd
import hssm
import arviz as az
import sqlite3
from datetime import datetime

In [ ]:
df_hssm = pd.read_csv('hssm_exp4_data_ab.csv')

In [ ]:
df_hssm['ab_nominal_binary'] = (df_hssm['ab_nominal'] == 10).astype(int)

In [ ]:
df_hssm

In [ ]:
def get_fitted_participants(db_path, table_name):
    with sqlite3.connect(db_path) as conn:
        try:
            q = f"SELECT DISTINCT participant_id FROM {table_name}"
            return set(pd.read_sql(q, conn)['participant_id'])
        except Exception:
            return set()


def write_summary_to_sql(df, db_path, table_name):
    df = df.copy()
    df["timestamp"] = datetime.now().isoformat()

    with sqlite3.connect(db_path) as conn:
        df.to_sql(table_name, conn, if_exists="append", index=False)


In [ ]:
def fit_hssm_mod_v_single(
    df, participant_id, participant_column,
    predictor='ab_nominal', use_log=False
):
    df = df.copy()

    df['X'] = (df[predictor] == 10).astype("float64")

    df_sub = (
        df[df[participant_column] == participant_id]
        .drop(columns=[participant_column])
    )

    print(f"___Participant {participant_id} | V only ___")
    print("Median RT =", np.median(df_sub['rt']))
    print("N trials =", len(df_sub))
    v_prior = {
        "Intercept": {"name": "Normal", "mu": 0.45, "sigma": 0.22},
        "X": {"name": "Normal", "mu": 0.0, "sigma": 0.15},
    }

    model = hssm.HSSM(
        data=df_sub,
        model="ddm",
        include=[
            {"name": "v", "formula": "v ~ 1 + X", "prior": v_prior},
        ],
    )

    idata = model.sample(
        cores=3,
        chains=3,
        draws=300,
        tune=1000,
        progressbar=True,
        target_accept=0.99,
    )

    summary_df = (
        az.summary(idata)
        .reset_index()
        .rename(columns={"index": "param"})
    )
    summary_df["participant_id"] = participant_id

    return summary_df

In [ ]:
def run_sequential_fits(
    df_hssm,
    participant_column,
    db_path,
    predictor='ab_nominal',
    use_log=False,
    max_participants=10,
    model_name=None,
):
    participants = df_hssm[participant_column].unique()

    models = {
        "ddm_mod_v":    fit_hssm_mod_v_single,
    }

    # If model_name specified, only run that model
    if model_name:
        models = {model_name: models[model_name]}

    for table_name, fit_func in models.items():
        print(f"\n===== Running model: {table_name} =====")

        fitted = get_fitted_participants(db_path, table_name)
        remaining = [p for p in participants if p not in fitted]

        print(f"{len(remaining)} participants remaining")

        fitted_count = 0

        for i, pid in enumerate(remaining, 1):
            if fitted_count >= max_participants:
                print(
                    f"\n⏸️  Reached {max_participants} participants "
                    f"for model {table_name}. Re-run to continue."
                )
                break

            print(f"\n--- {table_name}: participant {pid} ({i}/{len(remaining)}) ---")

            try:
                summary_df = fit_func(
                    df=df_hssm,
                    participant_id=pid,
                    participant_column=participant_column,
                    predictor=predictor,
                    use_log=use_log,
                )

                write_summary_to_sql(
                    summary_df,
                    db_path=db_path,
                    table_name=table_name,
                )

                fitted_count += 1

            except Exception as e:
                print(f"❌ Failed participant {pid}: {e}")
                continue


In [ ]:
# Fit the next 20 for ddm_mod_th
run_sequential_fits(
    df_hssm=df_hssm,
    participant_column="participant_id",
    db_path="hssm_fits.sqlite",
    predictor="ab_nominal",
    use_log=False,
    max_participants=50,
    model_name="ddm_mod_v",
)

In [2]:
import sqlite3
import pandas as pd

db_path = "hssm_fits.sqlite"
with sqlite3.connect(db_path) as conn:
    # Get all tables that exist
    tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    )
    print("Tables in database:")
    print(tables)

Tables in database:
           name
0  ddm_mod_th_v
1    ddm_mod_th
2     ddm_mod_v


In [3]:
with sqlite3.connect(db_path) as conn:
    # Read a specific table, e.g., 'ddm_mod_th_v'
    df_ddm_mod_th_v = pd.read_sql("SELECT * FROM ddm_mod_v;", conn)
    print("\nData from ddm_mod_th table:")
    n_df = pd.DataFrame(df_ddm_mod_th_v)


Data from ddm_mod_th table:


In [4]:
n_df

,param,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat,participant_id,timestamp
0,v_Intercept,0.478,0.148,0.194,0.746,0.006,0.004,672.0,625.0,1.00,708,2026-01-30T17:44:28.423188
1,v_X,0.084,0.124,-0.172,0.284,0.005,0.004,655.0,596.0,1.00,708,2026-01-30T17:44:28.423188
2,z,0.508,0.038,0.437,0.579,0.002,0.001,545.0,512.0,1.01,708,2026-01-30T17:44:28.423188
3,a,0.866,0.047,0.790,0.960,0.002,0.001,551.0,579.0,1.00,708,2026-01-30T17:44:28.423188
4,t,0.619,0.022,0.577,0.656,0.001,0.001,421.0,408.0,1.00,708,2026-01-30T17:44:28.423188
...,...,...,...,...,...,...,...,...,...,...,...,...
1710,v_Intercept,0.749,0.142,0.445,0.949,0.034,0.024,18.0,280.0,1.13,1113,2026-02-02T00:01:25.368211
1711,v_X,-0.041,0.134,-0.303,0.190,0.035,0.025,15.0,24.0,1.14,1113,2026-02-02T00:01:25.368211
1712,t,0.964,0.040,0.889,1.009,0.010,0.007,26.0,108.0,1.20,1113,2026-02-02T00:01:25.368211
1713,z,0.467,0.044,0.376,0.540,0.014,0.010,10.0,54.0,1.24,1113,2026-02-02T00:01:25.368211


In [5]:
len(n_df.participant_id.unique())

343

In [6]:
n_df[n_df['participant_id'] == 708]

,param,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat,participant_id,timestamp
0,v_Intercept,0.478,0.148,0.194,0.746,0.006,0.004,672.0,625.0,1.00,708,2026-01-30T17:44:28.423188
1,v_X,0.084,0.124,-0.172,0.284,0.005,0.004,655.0,596.0,1.00,708,2026-01-30T17:44:28.423188
2,z,0.508,0.038,0.437,0.579,0.002,0.001,545.0,512.0,1.01,708,2026-01-30T17:44:28.423188
3,a,0.866,0.047,0.790,0.960,0.002,0.001,551.0,579.0,1.00,708,2026-01-30T17:44:28.423188
4,t,0.619,0.022,0.577,0.656,0.001,0.001,421.0,408.0,1.00,708,2026-01-30T17:44:28.423188
